![Banner](../Image/03_DeepCuaslaML.png)

# 4.1 Neural Granger Causality Models


## Part 1: Conceptual Foundation

### Classical Granger Causality

A variable $X$ Granger-causes $Y$ if past values of $X$ significantly improve the forecast of $Y$ beyond what $Y$'s own past provides. For lag order $p$:

$$
Y_t = \sum_{k=1}^{p} A_k Y_{t-k} + \epsilon_t
$$

If the coefficient block connecting $X_{t-k} \rightarrow Y_t$ is nonzero, then $X$ Granger-causes $Y$. This is inherently linear and can miss nonlinear dynamics.

### Neural Granger Causality: The Big Idea

Replace the linear coefficients with neural networks while preserving sparse input selection logic.

> If a neural network trained to predict $Y_t$ assigns zero (or near-zero) weight to all lags of $X$, then $X$ does not Granger-cause $Y$.

Sparsity is enforced with group LASSO regularization on input weights, grouping all lags of a variable together.

## Part 2: The Four Models

### 1) cMLP (Component-wise MLP)

A separate MLP is trained for each target variable $Y_i$:

$$
\hat{Y}_{i,t} = \text{MLP}_i(Y_{1,t-1:t-p},\ldots,Y_{d,t-1:t-p})
$$

Training objective:

$$
\mathcal{L} = \underbrace{\text{MSE}}_{\text{prediction}} + \lambda \underbrace{\sum_j \lVert W_j^{(1)} \rVert_2}_{\text{group LASSO on input weights}}
$$

### 2) cLSTM (Component-wise LSTM)

Uses one LSTM per target variable. Better for sequential/long-lag dynamics:

$$
h_t = \text{LSTM}_i(Y_{1:d,\ t-p:t-1}), \quad \hat{Y}_{i,t} = W_{\text{out}} h_t
$$

### 3) ECONOMY-SRU

A structurally constrained recurrent unit with a soft gate matrix $G \in [0,1]^{d\times d}$:

- Learns sparse variable interactions
- Lower computational cost than full LSTMs

### 4) NRI (Neural Relational Inference)

NRI (Kipf et al., 2018) is a graph-VAE approach:

- **Encoder** infers latent edge distributions $q(z\mid X)$
- **Decoder** predicts trajectories from sampled edges

$$
\mathcal{L} = \mathbb{E}_{q(z\mid X)}[\log p(X\mid z)] - \mathrm{KL}[q(z\mid X)\|p(z)]
$$

In [1]:
import importlib
import subprocess
import sys

PACKAGES = [
    "numpy", "pandas", "scipy", "torch", "scikit-learn",
    "matplotlib", "seaborn", "networkx", "yfinance",
]

for pkg in PACKAGES:
    mod = "sklearn" if pkg == "scikit-learn" else pkg
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import pydeepcausalml  # noqa: F401
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/zia207/PyDeepCausalML.git"]
    )

import pydeepcausalml
print("pydeepcausalml", pydeepcausalml.__version__, "ready.")


pydeepcausalml 0.2.0 ready.


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from pydeepcausalml import set_seed

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

set_seed(42)
run_fast = True


torch: 2.10.0+cu128
cuda: True


### Data 

In [3]:
TICKERS = {
    "XLF": "Financials",
    "XLE": "Energy",
    "XLK": "Technology",
    "XLV": "Healthcare",
    "XLI": "Industrials",
    "XLY": "ConsumerDisc",
    "XLP": "ConsumerStap",
    "XLU": "Utilities",
}

import yfinance as yf

raw = yf.download(
    list(TICKERS.keys()),
    start="2018-01-01",
    end="2024-01-01",
    auto_adjust=True,
    progress=False,
)["Close"]

returns = np.log(raw / raw.shift(1)).dropna()
returns.columns = [TICKERS[t] for t in returns.columns]

if returns.shape[0] < 50:
    print("yfinance unavailable; using synthetic demo data.")
    rng = np.random.default_rng(42)
    T, d = 1500, len(TICKERS)
    VAR_NAMES = list(TICKERS.values())
    market = rng.normal(0.0, 0.8, size=(T, 1)).astype(np.float64)
    idio = rng.normal(0.0, 0.6, size=(T, d)).astype(np.float64)
    loading = np.linspace(0.5, 1.1, d, dtype=np.float64)[None, :]
    data_np = (market @ loading + idio).astype(np.float64)
else:
    data_np = returns.values.astype(np.float64)
    T, d = data_np.shape
    VAR_NAMES = returns.columns.tolist()

print(f"Shape: {data_np.shape}")
print(f"Variables ({d}): {VAR_NAMES}")
print(f"Time steps: {T}")


Shape: (1508, 8)
Variables (8): ['Energy', 'Financials', 'Industrials', 'Technology', 'ConsumerStap', 'Utilities', 'Healthcare', 'ConsumerDisc']
Time steps: 1508


### Data Preparation

In [4]:
LAG = 5
EPOCHS = 60 if run_fast else 120
BATCH_SIZE = 32
LR = 5e-4
LAM = 0.005
HIDDEN = 32
DEVICE = "cpu"

print(f"LAG={LAG}, EPOCHS={EPOCHS}, device={DEVICE}")


LAG=5, EPOCHS=60, device=cpu


### Define Models

In [5]:
from pydeepcausalml import (
    NeuralGrangerCMLP,
    NeuralGrangerCLSTM,
    neural_granger_model,
)

# cMLP and cLSTM live in pydeepcausalml.timeseries (group-sparse Granger models).
print("Neural Granger models imported from pydeepcausalml.")


Neural Granger models imported from pydeepcausalml.


### Structurally-constrained RNN for Granger causality

The following section introduces a structurally-constrained recurrent neural network (RNN) model designed for Granger causality analysis. In this architecture, the connections (or "causal links") between variables are explicitly parameterized and learnable. These constraints allow the model to represent, learn, and visualize which time series variables directly influence others,  making it interpretable with respect to causality structure.

In [6]:
from pydeepcausalml import NeuralGrangerEconomySRU, NeuralRelationalInference

# EconomySRU and NRI are extended neural Granger estimators in the same module family.
print("Extended neural Granger models ready.")


Extended neural Granger models ready.


### Train Models

In [7]:
def fit_granger(method: str):
    """Fit a neural Granger estimator via pydeepcausalml."""
    common = dict(
        lag=LAG, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
        device=DEVICE, random_state=42,
    )
    if method == "cmlp":
        est = neural_granger_model(
            method, hidden_dim=HIDDEN, lambda_group=LAM, **common,
        )
    elif method in ("clstm", "economysru"):
        est = neural_granger_model(
            method, hidden=HIDDEN, lambda_sparse=LAM, **common,
        )
    else:
        est = neural_granger_model(method, hidden=HIDDEN, **common)
    est.fit(data_np)
    return est


In [8]:
models = {}
histories = {}

for name, method in [
    ("cMLP", "cmlp"),
    ("cLSTM", "clstm"),
    ("ECONOMY-SRU", "economysru"),
    ("NRI", "nri"),
]:
    print(f"Training {name} ...")
    est = fit_granger(method)
    models[name] = est
    histories[name] = est.history_
    print(f"  done — final loss={est.history_['loss'][-1]:.4f}")


Training cMLP ...
  done — final loss=0.9467
Training cLSTM ...
  done — final loss=0.9391
Training ECONOMY-SRU ...
  done — final loss=1.0586
Training NRI ...
  done — final loss=1.1770


In [ ]:
print("\n" + "=" * 45)
print(f"{'Model':<15} {'Final loss':>12}")
print("=" * 45)
for name, hist in sorted(histories.items(), key=lambda x: x[1]["loss"][-1]):
    print(f"{name:<15} {hist['loss'][-1]:>12.6f}")
print("=" * 45)


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, name in zip(axes, ["cMLP", "cLSTM", "ECONOMY-SRU", "NRI"]):
    hist = histories[name]
    ax.plot(hist["loss"], lw=2)
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Loss")
plt.tight_layout()
plt.show()

# Causal matrices
def _causal_matrix(est):
    if hasattr(est, "get_scores"):
        return est.get_scores()
    return est.adjacency_matrix()

causal_mats = {name: _causal_matrix(est) for name, est in models.items()}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, mat) in zip(axes.flatten(), causal_mats.items()):
    sns.heatmap(
        mat, ax=ax, cmap="magma",
        xticklabels=VAR_NAMES, yticklabels=VAR_NAMES, cbar=True,
    )
    ax.set_title(f"{name} inferred causal strengths")
    ax.set_xlabel("Source variable")
    ax.set_ylabel("Target variable")
plt.tight_layout()
plt.show()


## Notes

- This notebook is intended as a practical tutorial, not a production-grade causal discovery pipeline.
- Financial returns are noisy and weakly causal; inferred links should be treated as exploratory.
- You can tune `LAG`, `LAM`, hidden size, and training epochs to trade off fit vs. sparsity.
- For stricter graph sparsity with NRI, consider annealing temperature and adding explicit edge sparsity priors.